In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Memory experiment analysis: min-distance + ROC curves.
Refactored into a single organized script. 
Modify only `run_experiment_at_noise` to explore new dynamics.
"""
# %load_ext autoreload
# %autoreload 2
# ===================== Imports =====================
import matplotlib
print(matplotlib.__file__)
import sys, os, glob, json, math, datetime
from collections import defaultdict

import pandas as pd
import numpy as np
import torch
#import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LinearRegression

# project-specific paths
sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

import DistanceMemoryModel
import encoders
from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used, load_results_with_exclusion_no_dropping
from utls.runners import run_experiment_scores, run_experiment_scores_itemwise, run_experiment_itemwise_hits_fas
from utls.analysis_helpers import rocs_across_noise, convert_human_to_model_struct, compute_scaling_vs_human, convert_human_to_model_struct
from utls.analysis_helpers import auroc_to_dprime, compute_model_dprime_curve
from utls.analysis_helpers import roc_for_isi, auroc_to_dprime, find_optimal_roc_threshold
from utls.plotting import plot_across_noise, plot_noise_overlays, plot_histograms_all_models, plot_model_grid_summary, plot_groupwise_item_response_scatter, plot_model_decay_summary
from utls.io_utils import make_model_save_dir, save_all_figures, save_single_figure, load_all_runs, save_runs_summary

import os, json
import numpy as np
from scipy.stats import norm
from sklearn.utils import resample
from utls.roc_utils import roc_from_arrays  # if separate, or inline

import math


import numpy as np
import torch
import pandas as pd
from collections import defaultdict



In [ ]:
save_base="/om2/user/bjmedina/auditory-memory/memory/figures/model-behavior-const-step-v3/"
t_results = glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/*V15.csv")

PC_DIMS = 256
DEVICE = 'cuda'
NV_DEFAULT = 0.1

# Encode clean reps
pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=PC_DIMS, model_params=model_params,
    sr=20000, rms_level=0.05, duration=2.0, device=DEVICE
)

!echo $CUDA_VISIBLE_DEVICES

In [ ]:
# OLD MULTI-ISI DATA
tasks = ["env-sounds" ,"glob-music", "atexts", "nhs-region-len120"]
which_task = tasks[2] # "global-music-len120", "atexts-len120" "nhs-region-len120"
base_path_ = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/len120_multi/"
seqs_paths = {"env-sounds": "mem_exp_ind-nature_2025", 
              "glob-music": "global-music-2025-n_80",
              "atexts": "mem_exp_atexts_2025",
              "nhs-region-len120": "nhs-region-n_80"}


hr_task_name = {"env-sounds": "Industrial and Nature", 
              "glob-music": "Globalized Music",
              "atexts": "Auditory Textures",
              "nhs-region-len120": " 'Natural History of Song' "}

exps_, seqs_, fnames, skipped_exps, skipped_seqs, skipped_fnames = load_results_with_exclusion_no_dropping(f"/mindhive/mcdermott/www/bjmedina/experiments/{which_task}/results/{which_task}/len120_multi",
                                                    min_dprime=2,
                                                    min_trials=120,
                                                    skip_len60=True,
                                                    verbose=True,
                                                    return_skipped=True)

# Build experiment_list
experiment_list_ = []
for seq_ in seqs_:
    with open(base_path_.format(which_task) + seq_, 'r') as f:
        data = json.load(f)
    stim_paths = [base_path_.format(which_task) + s 
                  for s in data['filenames_order']]

    experiment_list_.append(stim_paths)

all_files_ = sorted({fn for seq in experiment_list_ for fn in seq})
name_to_idx_ = {fn: i for i, fn in enumerate(all_files_)}

tmp = DistanceMemoryModel.DistanceMemoryModel(pc_texture_model, NV_DEFAULT, criterion=0.0, device=DEVICE)
tmp._fill_memory_bank(all_files_)
with torch.no_grad():
    X0_ = torch.stack([rep.detach().clone().view(-1) for rep in tmp.memory_bank], dim=0).to(DEVICE)

In [ ]:
# SINGLE ISI DATA
which_task = "atexts-len120"

seqs_paths = {
    "ind-nature-len120": "mem_exp_ind-nature_2025", 
    "global-music-len120": "global-music-2025-n_80",
    "atexts-len120": "mem_exp_atexts_2025",
    "nhs-region-len120": "nhs-region-n_80"
}
base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"
exps, seqs, fnames = load_results_with_exclusion_no_dropping(
    f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
    min_dprime=2, min_trials=120, skip_len60=True, verbose=False, return_skipped=False
)
move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)

# Build experiment_list
experiment_list = []
for seq in seqs:
    with open(base_path.format(seqs_paths[which_task]) + seq, 'r') as f:
        data = json.load(f)
    stim_paths = ["/mindhive/mcdermott/www/mturk_stimuli/bjmedina/" + seqs_paths[which_task] + "/" + s 
                  for s in data['filenames_order']]
    experiment_list.append(stim_paths)

all_files = sorted({fn for seq in experiment_list for fn in seq})
name_to_idx = {fn: i for i, fn in enumerate(all_files)}

tmp = DistanceMemoryModel.DistanceMemoryModel(pc_texture_model, NV_DEFAULT, criterion=0.0, device=DEVICE)
tmp._fill_memory_bank(all_files)
with torch.no_grad():
    X0 = torch.stack([rep.detach().clone().view(-1) for rep in tmp.memory_bank], dim=0).to(DEVICE)

In [ ]:
human_runs = []
for result in t_results:
    df = pd.read_csv(result)
    main_exp = df[df['stim_type'] == 'main']
    if len(main_exp) < 256:
        continue
    human_runs.append(convert_human_to_model_struct(main_exp))

# Compute average human d′ vs ISI

isis_human = [0, 1, 2, 3, 4, 8, 16, 32, 64]
dprimes_human = []
for k in isis_human:
    aucs = []
    for run in human_runs:
        res = roc_for_isi(run, k)
        if res is not None:
            _, _, auc = res
            aucs.append(auroc_to_dprime(auc))
    dprimes_human.append(np.nanmean(aucs))
    
human_curve = np.array(dprimes_human, dtype=float)
human_curve = human_curve[~np.isnan(human_curve)]
print(human_curve)

In [ ]:
import pickle, glob, os

savedir = "/om2/user/bjmedina/auditory-memory/memory/figures/model-behavior-const-step-v3/"
all_data = load_all_runs(savedir)
print(f"Loaded {len(all_data)} models.")

In [ ]:
isis = [0, 1, 2, 3, 4, 8, 16, 32, 64]
human_curve = {
    "isi": np.array(isis_human),
    "dprime": np.array(dprimes_human)
}

best_fits = plot_model_decay_summary(all_data, human_curve, isis, savedir, returns=True)

In [ ]:
best_fit_parameters = list(best_fits.keys())

# pick the (metric, rate) with the highest combined score
best_key = max(best_fits, key=lambda k: best_fits[k]['fit']['score'])
best_model = best_fits[best_key]

print(f"Global best model: {best_key}")
print(f"Noise σ₀={best_model['noise']}, "
      f"ρ={best_model['fit']['rho']:.3f}, "
      f"Δd′={best_model['fit']['drop']:.2f}, "
      f"NMSE={best_model['fit']['nmse']:.3f}, "
      f"Score={best_model['fit']['score']:.3f}")

In [ ]:
best_decisions = {}

for key, entry in best_fits.items():
    metric, rate = key
    best_nv  = entry["noise"]
    runs     = entry["runs"]
    run_data = runs[best_nv]
    score_type = run_data.get("score_type", "distance")

    hits = np.asarray(run_data["hits"], float)
    fas  = np.asarray(run_data["fas"], float)
    if hits.size == 0 or fas.size == 0:
        continue

    print(entry['fit'].keys())
    roc_info = find_optimal_roc_threshold(hits, fas, score_type=score_type)

    if score_type == "distance":
        decision_threshold = -roc_info["threshold"]
    else:
        decision_threshold = roc_info["threshold"]

    best_decisions[key] = dict(
        noise=best_nv,
        metric=metric,
        rate=rate,
        encoder="texture_statistics",
        threshold=decision_threshold,
        fpr=roc_info["fpr"],
        tpr=roc_info["tpr"],
        dist=roc_info["distance"],
        rho=entry["fit"]["rho"],
        nmse=entry["fit"]["nmse"],
        score=entry["fit"]["score"],
        drop=entry["fit"]["drop"]
    )

    print(f"[{metric} | rate={rate}] "
          f"σ₀={best_nv:g}, nmse²={entry['fit']['nmse']:.3f}, "
          f"thr={decision_threshold:.3f}, FPR={roc_info['fpr']:.3f}, TPR={roc_info['tpr']:.3f}")

In [ ]:
import os
import pickle


force_rerun = False  # 👈 set to True to recompute and overwrite everything

best_results = {}

for key, best_model in best_decisions.items():
    # --- Parse model info ---
    parsed = best_model["metric"].split("+")
    metric, noise_mode = parsed[0], parsed[1]
    model_name = (
        "DistanceMemoryModel"
        if metric in {"euclidean", "cosine", "mahalanobis", "manhattan"}
        else "LikelihoodMemoryModel"
    )

    model_info = dict(
        model_name=model_name,
        metric=metric,
        noise_mode=noise_mode,
        encoder=best_model["encoder"],
        rate=best_model["rate"],
        run_id="prolific_batch"
    )

    save_dir = make_model_save_dir(savedir, model_info, which_task)
    pkl_path = os.path.join(save_dir, f"{which_task}-runs_single-isi.pkl")
    # --- If results already exist, load them instead of rerunning ---
    if os.path.exists(pkl_path) and not force_rerun:
        print(f"📂 Loading existing {metric}+{noise_mode} (rate={best_model['rate']})")

        with open(pkl_path, "rb") as f:
            results_model = pickle.load(f)

        best_results[key] = {"itemwise_responses": results_model}
        continue

    # --- Otherwise, run simulation ---
    print(f"🚀 Running model: {metric}+{noise_mode} (rate={best_model['rate']})")

    results_model = run_experiment_itemwise_hits_fas(
        sigma0=best_model["noise"],
        rate=best_model["rate"],
        metric=metric,
        noise_mode=noise_mode,
        X0=X0,
        name_to_idx=name_to_idx,
        experiment_list=experiment_list[:len(experiment_list)//2],
        decision_threshold=best_model["threshold"],
        debug=False,
    )

    print(results_model.keys())

    results = {"itemwise_responses": results_model}
    best_results[key] = results

    # --- Save outputs ---
    save_runs_summary(results_model, model_info, save_dir, which_task, single_isi=True)

print(f"\n✅ Finished loading/running {len(best_results)} total models.")

In [ ]:
#best_results[('cosine+linear-step', 0.5)].keys()
best_decisions[best_key]

In [ ]:
from utls.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability
from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir

from utls.reliability import compute_power_curve
from utls.plotting import plot_power_curve

import matplotlib.pyplot as plt

results_humans = compute_itemwise_split_half_reliability(exps, min_isi=16, max_isi=16)
hits = results_humans['itemwise_responses']['hits']
false_alarms  = results_humans['itemwise_responses']['false_alarms']

import numpy as np
from scipy.stats import norm

def compute_itemwise_dprime(df_hits, df_fas, eps=1e-5):
    # Ensure identical column sets
    common = sorted(set(df_hits.columns) & set(df_fas.columns))
    df_hits = df_hits[common]
    df_fas  = df_fas[common]

    hit_rates = df_hits.mean(axis=0).clip(eps, 1-eps)
    fa_rates  = df_fas.mean(axis=0).clip(eps, 1-eps)

    # Force pandas Series output
    dprime_values = norm.ppf(hit_rates.values) - norm.ppf(fa_rates.values)
    return pd.Series(dprime_values, index=common)

dprime = compute_itemwise_dprime(hits, false_alarms)

dprime.sort_values().plot(kind='bar', figsize=(18,6))
plt.ylabel("d'")
plt.xlabel("Stimulus")
plt.title("Per-sound d'")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import pearsonr, spearmanr
import numpy as np
import pandas as pd


def compute_intergroup_correlation(df_group1, df_group2, method="pearson", min_overlap=10, jitter_std=1e-2):
    """
    Compute intergroup correlation between two groups' itemwise mean responses.

    Adds small bounded jitter if one group's responses are constant.

    Args:
        df_group1, df_group2 : pd.DataFrame
            Rows = participants, columns = item IDs, values = binary or scalar responses.
        method : str
            'pearson' (default) or 'spearman'
        min_overlap : int
            Minimum number of overlapping items to compute a correlation.
        jitter_std : float
            Std dev of jitter to add if one group has zero variance.
    
    Returns:
        (r, n_items, item_df)
    """
    # align items
    common_items = sorted(set(df_group1.columns) & set(df_group2.columns))
    if len(common_items) < min_overlap:
        raise ValueError(f"Not enough overlapping items (found {len(common_items)})")

    # compute itemwise means
    g1_means = df_group1[common_items].mean(axis=0).to_numpy()
    g2_means = df_group2[common_items].mean(axis=0).to_numpy()

    # add bounded jitter if needed
    def add_jitter(vec):
        jitter = np.random.normal(0, jitter_std, size=vec.shape)
        return np.clip(vec + jitter, 0, 1)

    if np.allclose(np.std(g1_means), 0):
        g1_means = add_jitter(g1_means)
    if np.allclose(np.std(g2_means), 0):
        g2_means = add_jitter(g2_means)

    # compute correlation
    if method == "pearson":
        r, _ = pearsonr(g1_means, g2_means)
    elif method == "spearman":
        r, _ = spearmanr(g1_means, g2_means)
    else:
        raise ValueError("method must be 'pearson' or 'spearman'")

    item_df = pd.DataFrame({
        "item": common_items,
        "group1": g1_means,
        "group2": g2_means
    })

    return r, len(common_items), item_df

In [ ]:
r, common_items, item_df =compute_intergroup_correlation(hits, false_alarms)
print(r)

In [ ]:
import math 
import matplotlib.pyplot as plt

summary = []

for (metric_combo, rate) in best_results:
    entry = best_results[(metric_combo, rate)]

    # inside your intergroup reliability loop or preprocessing:
    if "loglikelihood" in metric_combo:
        # flip the sign of responses so higher = stronger "no"
        entry["itemwise_responses"]["hits"] *= -1
        entry["itemwise_responses"]["fas"]  *= -1

    r, n_items, item_df = compute_intergroup_correlation(
        hits,
        entry['itemwise_responses']['hits'],
        method="spearman"
    )

    if math.isnan(r):
        plot_groupwise_item_response_scatter(
            entry,
            title="Group-wise Item Responses",
            kind="hits",
            seed=42,
            split_method="half",
            save_path=None
        )
        
    r_fa, _, _ = compute_intergroup_correlation(
        false_alarms,
        entry['itemwise_responses']['fas'],
        method="spearman"
        
    )

    if math.isnan(r_fa):
        plot_groupwise_item_response_scatter(
            entry,
            title="Group-wise Item Responses",
            kind="fas",
            seed=43,
            split_method="half",
            save_path=None
        )
    
    summary.append({
        "metric_combo": metric_combo,
        "rate": rate,
        "r_hit": r,
        "r_fa": r_fa
    })

# --- Make DataFrame ---
df = pd.DataFrame(summary).sort_values(["metric_combo", "rate"])

# --- Plot ---
plt.figure(figsize=(18, 8))
x_labels = [f"{m}\nrate={r:g}" for m, r in zip(df["metric_combo"], df["rate"])]
x = np.arange(len(x_labels))

plt.plot(x, df["r_hit"], 'o-', color='green', label="Hits (Spearman r)")
plt.plot(x, df["r_fa"],  's--', color='red',   label="False Alarms (Spearman r)")

y_all = np.concatenate([
    df["r_hit"].to_numpy(dtype=float),
    df["r_fa"].to_numpy(dtype=float)
])
y_all = y_all[np.isfinite(y_all)]

if len(y_all) > 0:
    ymin = max(-0.1, np.nanmin(y_all) - 0.05)
    ymax = min(1.05, np.nanmax(y_all) + 0.05)
    plt.ylim(ymin, ymax)
else:
    plt.ylim(-0.1, 1.05)


plt.xticks(x, x_labels, rotation=45, fontsize=5, ha="right")
plt.ylabel("Intergroup Correlation (Spearman r)")
plt.title("Model–Human Intergroup Correlation per Metric × Rate")
plt.ylim(-0.4, 0.55)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(savedir + "/model-human_intergroup-correlation.png")
plt.show()
plt.close()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# === Compute NMSE–intergroup relationships ===
nmse_list, r_hit_list, r_fa_list = [], [], []

for (metric_combo, rate) in df[["metric_combo", "rate"]].itertuples(index=False):
    nmse = best_decisions[(metric_combo, rate)]["nmse"]
    nmse_list.append(nmse)
    
    row = df[(df["metric_combo"] == metric_combo) & (df["rate"] == rate)]
    r_hit_list.append(float(row["r_hit"]))
    r_fa_list.append(float(row["r_fa"]))

# Convert to numpy arrays
nmse_arr = np.array(nmse_list)
r_hit_arr = np.array(r_hit_list)
r_fa_arr  = np.array(r_fa_list)

# Compute Spearman correlations
rho_hit, _ = spearmanr(nmse_arr, r_hit_arr)
rho_fa,  _ = spearmanr(nmse_arr, r_fa_arr)
print(f"Spearman correlation (NMSE vs r_hit): {rho_hit:.2f}")
print(f"Spearman correlation (NMSE vs r_fa):  {rho_fa:.2f}")

# === Plot with noise ceilings ===
ceiling_hit = results_humans['split_half_reliability']['hits'][0]
ceiling_fa  = results_humans['split_half_reliability']['false_alarms'][0]

plt.figure(figsize=(6,3))

# --- Hits subplot ---
plt.subplot(1,2,1)
plt.scatter(nmse_arr, r_hit_arr, c="green", edgecolor="black", s=45, label=f"ρ={rho_hit:.2f}")
# plt.axhline(ceiling_hit, color="green", ls="--", lw=1.5, label=f"Noise ceiling = {ceiling_hit:.2f}")
plt.xlabel("NMSE (↓ better)")
plt.ylabel("Intergroup r (hits)")
plt.title("NMSE vs r_hit")
plt.ylim(-0.2, 1.05)
plt.grid(alpha=0.3)
plt.legend(frameon=False, fontsize=8)

# --- False alarms subplot ---
plt.subplot(1,2,2)
plt.scatter(nmse_arr, r_fa_arr, c="red", edgecolor="black", s=45, label=f"ρ={rho_fa:.2f}")
# plt.axhline(ceiling_fa, color="red", ls="--", lw=1.5, label=f"Noise ceiling = {ceiling_fa:.2f}")
plt.xlabel("NMSE (↓ better)")
plt.ylabel("Intergroup r (false alarms)")
plt.title("NMSE vs r_fa")
plt.ylim(-0.2, 1.05)
plt.grid(alpha=0.3)
plt.legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
entry = best_results[best_key]
hits_model = entry['itemwise_responses']['hits']
false_alarms_model = entry['itemwise_responses']['fas']


dprime_model = compute_itemwise_dprime(hits_model, false_alarms_model)

dprime_model.sort_values().plot(kind='bar', figsize=(18,6))
plt.ylabel("d'")
plt.xlabel("Stimulus")
plt.title("Per-sound d'")
plt.tight_layout()
plt.show()

In [ ]:
def top_n_from_series(s, n=40):
    """Return the top-n index labels from a Series sorted by value descending."""
    # For a Series, sort_values takes only keyword arguments
    return s.sort_values(ascending=False).head(n).index.tolist()

def intersection_top_n(series_humans, series_model, n=40):
    """Return intersection & count for two Series of d′ values."""
    top_h = set(top_n_from_series(series_humans, n))
    top_m = set(top_n_from_series(series_model, n))

    inter = sorted(top_h & top_m)
    return inter, len(inter)

# ---- RUN IT ----
n = 40
inter, k = intersection_top_n(dprime, dprime_model, n=n)

print("overlap (%):", k/n)
print(inter)


In [ ]:
import pandas as pd

import librosa
import numpy as np

stim_path = f"/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{seqs_paths[which_task]}"
def compute_spectral_centroid(path):
    y, sr = librosa.load(path, sr=None)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    return centroid.mean()

spectral_centroids = {
    stim: compute_spectral_centroid(f"{stim_path}/{stim}")
    for stim in dprime.index
}

features_df = pd.DataFrame.from_dict(spectral_centroids, orient='index', columns=["spec_centroid"])

df = pd.DataFrame({
    "dprime_human": dprime,
    "dprime_model": dprime_model,
})

# Join with acoustic features
df = df.join(features_df, how="inner")



In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import scipy.stats as st
from scipy.signal import hilbert

# ==============================================================
# Utility functions
# ==============================================================

def safe_load(path, sr=None):
    """Load audio with safety wrapper."""
    try:
        y, sr = librosa.load(path, sr=sr)
        if len(y) == 0:
            return None, None
        return y, sr
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None, None


def spectral_entropy(y, sr, n_fft=2048, hop=512):
    """Compute spectral entropy over frames."""
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop))
    ps = S / np.sum(S, axis=0, keepdims=True)
    entropy = -np.sum(ps * np.log(ps + 1e-12), axis=0)
    return float(np.mean(entropy))


def temporal_envelope(y):
    """Amplitude envelope via Hilbert transform."""
    analytic = hilbert(y)
    env = np.abs(analytic)
    return env


def modulation_spectrum(env, sr, lowpass=32):
    """
    Compute modulation spectrum up to 32 Hz.
    Returns 8 modulation energy bands.
    """
    hop = 1
    env = env[::hop]
    fft = np.abs(np.fft.rfft(env))
    freqs = np.fft.rfftfreq(len(env), 1/sr/hop)
    out = []
    bands = [(1,2), (2,4), (4,8), (8,16), (16,32)]
    for lo, hi in bands:
        m = fft[(freqs>=lo)&(freqs<hi)]
        out.append(float(np.sum(m)))
    return out


def modulation_entropy(env):
    """Entropy of the envelope signal."""
    hist, _ = np.histogram(env, bins=50, density=True)
    hist += 1e-12
    return -np.sum(hist * np.log(hist))


def compute_hnr(y, sr):
    """Compute Harmonic-to-Noise Ratio (HNR)."""
    try:
        f0, voiced, prob = librosa.pyin(y, sr=sr, fmin=50, fmax=2000)
        hnr = np.nanmean(prob)
        return float(hnr)
    except:
        return np.nan


# ==============================================================
# MAIN FEATURE EXTRACTION PIPELINE
# ==============================================================

def extract_features_single(path):
    """Extract ~50 features from a single waveform."""
    y, sr = safe_load(path, sr=None)
    if y is None:
        return None

    # ========== Low-level spectral ==========
    centroid     = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    bandwidth    = float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)))
    rolloff      = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
    flatness     = float(np.mean(librosa.feature.spectral_flatness(y=y)))
    contrast     = float(np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)))
    rms          = float(np.mean(librosa.feature.rms(y=y)))
    avg_pitch    = float(np.median(librosa.yin(y=y, fmin=20, fmax=20000, sr=sr)))
    spec_entropy = spectral_entropy(y, sr)

    # ========== Temporal ==========
    env = temporal_envelope(y)
    env_var = float(np.var(env))
    env_mean = float(np.mean(env))
    temp_centroid = float(np.mean(librosa.feature.spectral_centroid(y=env, sr=sr)))
    attack = float(np.argmax(env > 0.5 * np.max(env)) / sr)

    # ========== Modulation (5 bands) ==========
    mod_bands = modulation_spectrum(env, sr)
    mod_ent = modulation_entropy(env)

    # ========== MFCCs ==========
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_means = [float(np.mean(c)) for c in mfcc]

    # ========== Harmonicity (HNR) ==========
    hnr = compute_hnr(y, sr)

    # Combine into dict
    features = {
        "spec_centroid": centroid,
        "spec_flatness": flatness,
        "spec_contrast": contrast,
        "rms": rms,
        "spec_entropy": spec_entropy,
        "env_var": env_var,
        "env_mean": env_mean,
        "temp_centroid": temp_centroid,
        "mod_1_2": mod_bands[0],
        "mod_2_4": mod_bands[1],
        "mod_4_8": mod_bands[2],
        "mod_8_16": mod_bands[3],
        "mod_16_32": mod_bands[4],
    }

    return features


def extract_all_features(paths_dict):
    """
    Extract a rich set of audio features for all sounds.
    paths_dict: { stim_name : full_path }

    Returns:
        DataFrame with rows = stimuli, cols = features
    """
    rows = []
    for stim, path in paths_dict.items():
        print(f"Extracting features for {stim} ...")
        feats = extract_features_single(path)
        if feats is None:
            continue
        feats["stimulus"] = stim
        rows.append(feats)

    df = pd.DataFrame(rows)
    df = df.set_index("stimulus")
    return df

In [ ]:
paths = {
    stim: f"{stim_path}/{stim}"
    for stim in dprime.index
}

features_df = extract_all_features(paths)

print(features_df.head())
print(features_df.shape)

In [ ]:
def plot_feature_grid(
    features_df,
    target1,
    target2=None,
    target1_name="target1",
    target2_name="target2",
    color1="blue",
    color2="magenta",
    max_cols=5,
    figsize=(20, 20),
    sort_by="corr_abs",   # options: "corr_abs", "corr_signed", None
    save_path=None
):
    """
    General scatter-grid plotter for features vs target values.
    
    Args:
        features_df: DataFrame indexed by item, columns=features
        target1: Series indexed by item (e.g., human d', hit-rate)
        target2: Optional Series indexed by item (e.g., model d')
        target1_name: Label for target1 in the legend/titles
        target2_name: Label for target2
        color1, color2: scatterplot colors
        max_cols: number of columns in the grid
        figsize: overall figure size
        sort_by:
            "corr_abs"    → sort by absolute correlation w/ target1
            "corr_signed" → signed correlation sort
            None          → do not sort features
        save_path: if provided, saves PNG to this path
        
    Returns:
        human_corrs (dict), model_corrs (dict or None)
    """

    # --- Alignment ---
    common = features_df.index.intersection(target1.index)
    if target2 is not None:
        common = common.intersection(target2.index)

    feats = features_df.loc[common]
    t1 = target1.loc[common]

    if target2 is not None:
        t2 = target2.loc[common]

    # --- Compute correlations with target1 ---
    human_corrs = {
        feat: np.corrcoef(feats[feat].values, t1.values)[0, 1]
        for feat in feats.columns
    }

    # --- Optional: compute correlations with target2 ---
    if target2 is not None:
        model_corrs = {
            feat: np.corrcoef(feats[feat].values, t2.values)[0, 1]
            for feat in feats.columns
        }
    else:
        model_corrs = None

    # --- Sort features ---
    if sort_by == "corr_abs":
        sorted_feats = sorted(human_corrs.keys(), key=lambda f: abs(human_corrs[f]), reverse=True)
    elif sort_by == "corr_signed":
        sorted_feats = sorted(human_corrs.keys(), key=lambda f: human_corrs[f], reverse=True)
    else:
        sorted_feats = list(features_df.columns)

    # --- Grid size ---
    n_features = len(sorted_feats)
    n_cols = max_cols
    n_rows = int(np.ceil(n_features / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten()

    # --- Plot each feature ---
    for i, feat in enumerate(sorted_feats):
        ax = axes[i]
        x = feats[feat].values

        # correlations
        r1 = human_corrs[feat]
        ax.scatter(x, t1.values, s=10, color=color1, alpha=0.7, label=target1_name)

        title = f"{feat}\n r_{target1_name}={r1:.2f}"

        # plot second target if provided
        if target2 is not None:
            r2 = model_corrs[feat]
            ax.scatter(x, t2.values, s=10, color=color2, alpha=0.7, label=target2_name)
            title += f", r_{target2_name}={r2:.2f}"

        ax.set_title(title, fontsize=9)
        ax.set_xlabel(feat)
        ax.set_ylabel(target1_name)

    # --- Remove unused axes ---
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    fig.tight_layout()

    # --- Save if requested ---
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300)
        print(f"Saved figure to: {save_path}")

    plt.show()

    return human_corrs, model_corrs

In [ ]:
def extract_texture_stats_all(paths_dict, pc_texture_model, feature_names=None):
    """
    Run pc_texture_model(path) for every stimulus and stack results into a DataFrame.
    
    paths_dict: {stimulus_name: filepath}
    pc_texture_model: callable, returns 1D array-like of features for a given path
    feature_names: optional list of names for each feature dimension
    
    Returns:
        DataFrame with index=stimulus, columns=features
    """
    rows = {}
    n_features = None

    for stim, path in paths_dict.items():
        stats = pc_texture_model(path)          # 1D array-like
        stats = np.asarray(stats.cpu()).ravel()       # ensure 1D

        if n_features is None:
            n_features = len(stats)
            if feature_names is None:
                feature_names = [f"tex_{i}" for i in range(n_features)]
        else:
            if len(stats) != n_features:
                raise ValueError(f"{stim} has {len(stats)} features, expected {n_features}")

        rows[stim] = stats

    df = pd.DataFrame.from_dict(rows, orient="index", columns=feature_names)
    return df

text_stats = extract_texture_stats_all(paths, pc_texture_model)

In [ ]:
def compute_texture_typicality_from_stats(texture_df):
    """
    Compute typicality directly in texture-statistics space:
    - normalize each feature by its std across sounds
    - compute Euclidean distance from the mean (in z-scored space)
    """
    # Per-feature mean and std
    mean = texture_df.mean(axis=0)
    std  = texture_df.std(axis=0).replace(0, np.nan)

    # Z-score (center + scale by std)
    z = (texture_df - mean) / std

    # Distance from origin in z-space == distance from mean in normalized space
    dists = np.linalg.norm(z.values, axis=1)

    typicality = pd.Series(dists, index=texture_df.index, name="texture_typicality")
    return typicality, z

texture_df = compute_texture_typicality_from_stats(text_stats)

In [ ]:
def compute_knn_median_distance(texture_df, k=5, metric="euclidean"):
    """
    Compute typicality using median distance to k nearest neighbors:
    
        - Normalize each feature by its std across sounds
        - Compute KNN distances
        - Atypicality = median distance to k-NN
        - Typicality = -Atypicality (higher = more typical)
    
    Args:
        texture_df : DataFrame (sounds × features)
        k          : number of neighbors
        metric     : distance metric (euclidean or cosine)

    Returns:
        typicality      : Series (higher = more typical)
        atypicality     : Series (higher = more atypical)
        z_features      : DataFrame (z-scored feature space)
    """
    from sklearn.neighbors import NearestNeighbors
    # --- 1. Normalize by per-feature std ---
    std = texture_df.std(axis=0).replace(0, np.nan)
    z = texture_df / std  # NOTE: no mean-centering (as requested)

    X = z.values

    # --- 2. KNN distances ---
    knn = NearestNeighbors(n_neighbors=k+1, metric=metric).fit(X)
    dists, _ = knn.kneighbors(X)

    # dists[:, 0] is self-distance (0), ignore it
    median_knn = np.median(dists[:, 1:], axis=1)

    # --- 3. Typicality and atypicality ---
    atypicality = pd.Series(median_knn, index=texture_df.index, name=f"knn{k}_median_atypicality")
    typicality  = pd.Series(-median_knn,   index=texture_df.index, name=f"knn{k}_median_typicality")

    return typicality, atypicality, z

texture_typicality, raw_texture_df = texture_df
typicality, atypicality, z = compute_knn_median_distance(raw_texture_df, k=5, metric='euclidean')

In [ ]:
features_df["texture_typicality_knn5"] = typicality

In [ ]:
plot_feature_grid(
    features_df=features_df,
    target1=dprime,
    target2=dprime_model,
    target1_name="Human d'",
    target2_name="Model d'",
    max_cols=3,           # grid width
    figsize=(30, 20),
    save_path=savedir+f"{which_task}-feature_vs_dprime.png"# controls overall size
);


In [ ]:
hits_model = entry['itemwise_responses']['hits']

eps = 1e-5
hit_rates = hits.mean(axis=0).clip(eps, 1-eps)
fa_rates  = false_alarms.mean(axis=0).clip(eps, 1-eps)

hit_rates_model = hits_model.mean(axis=0).clip(eps, 1-eps)
fa_rates_modl  = false_alarms_model.mean(axis=0).clip(eps, 1-eps)

plot_feature_grid(
    features_df=features_df,
    target1=hit_rates,
    target2=hit_rates_model,
    color1="springgreen",
    color2="lightseagreen",
    target1_name="Human hit rate",
    target2_name="Model hit rate",
    max_cols=3,           # grid width
    figsize=(30, 20),
    save_path=savedir+f"{which_task}-feature_vs_hit.png"# controls overall size
);

In [ ]:
plot_feature_grid(
    features_df=features_df,
    target1=fa_rates,
    target2=fa_rates_modl,
    color1="maroon",
    color2="peru",
    target1_name="Human false alarm rate",
    target2_name="Model false alarm rate",
    max_cols=3,           # grid width
    figsize=(30, 20),
    save_path=savedir+f"{which_task}-feature_vs_fa.png"# controls overall size
);

In [ ]:
aligned_humans = (dprime - dprime.mean()) / dprime.std()
aligned_model = (dprime_model - dprime_model.mean()) / dprime_model.std()

r_hm = np.corrcoef(aligned_humans, aligned_model)[0,1]
print("Correlation: aligned humans vs aligned model =", r_hm)

residual = aligned_humans - aligned_model

aligned_humans = (dprime - dprime.mean()) / dprime.std()
aligned_model  = (dprime_model - dprime_model.mean()) / dprime_model.std()

residual = (aligned_humans - aligned_model).dropna()

# Sort by absolute disagreement
residual_sorted = residual.reindex(residual.abs().sort_values(ascending=False).index)
residual_sorted_nonabs = residual.reindex(residual.sort_values(ascending=False).index)


plt.figure(figsize=(18,6))

colors = ['tab:blue' if r > 0 else 'tab:red' for r in residual_sorted]

plt.bar(residual_sorted.index, residual_sorted.values, color=colors)
plt.xticks(rotation=90)

plt.ylabel(f"Human–Model Residual (z-scored d′)")
plt.title(
    f"Residuals Between Human and Model d′ (Aligned)  [r={r_hm:.2f}]\n"
    "Blue = Humans outperform model predictions | Red = Model overpredicts human performance"
)

plt.axhline(0, color='black', linewidth=1)
plt.tight_layout()
plt.savefig(savedir+"human_model_dprime_residuals.png");
plt.show()



In [ ]:
top10 = residual_sorted_nonabs.head(10)
bottom10 = residual_sorted_nonabs.tail(10)

print("Largest positive residuals (humans > model):")
print(top10)

print("\nLargest negative residuals (model > humans):")
print(bottom10)

top_items = residual_sorted_nonabs.head(10).index
bottom_items = residual_sorted_nonabs.tail(10).index

features_z = (features_df - features_df.mean()) / features_df.std()

top_feats_z = features_z.loc[top_items]
bottom_feats_z = features_z.loc[bottom_items]

diff_z = top_feats_z.mean() - bottom_feats_z.mean()
diff_z_sorted = diff_z.sort_values(ascending=False)
print(diff_z_sorted)

In [ ]:
aligned_humans = (hit_rates - hit_rates.mean()) / hit_rates.std()
aligned_model = (hit_rates_model - hit_rates_model.mean()) / hit_rates_model.std()
r_hm = np.corrcoef(aligned_humans, aligned_model)[0,1]

residual = (aligned_humans - aligned_model).dropna()

# Sort by absolute disagreement
residual_sorted = residual.reindex(residual.abs().sort_values(ascending=False).index)

plt.figure(figsize=(18,6))

colors = ['tab:blue' if r > 0 else 'tab:red' for r in residual_sorted]

plt.bar(residual_sorted.index, residual_sorted.values, color=colors)
plt.xticks(rotation=90)

plt.ylabel("Human–Model Residual (z-scored hit rates)")
plt.title(
    f"Residuals Between Human and Model hit rates (Aligned) [r={r_hm:.2f}]\n"
    "Blue = Humans outperform model predictions | Red = Model overpredicts human performance"
)

plt.axhline(0, color='black', linewidth=1)
plt.tight_layout()
plt.savefig(savedir+"human_model_hit_rate_residuals.png");
plt.show()



In [ ]:
aligned_humans = (fa_rates - fa_rates.mean()) / fa_rates.std()
aligned_model = (fa_rates_modl - fa_rates_modl.mean()) / fa_rates_modl.std()
r_hm = np.corrcoef(aligned_humans, aligned_model)[0,1]

residual = (aligned_humans - aligned_model).dropna()

# Sort by absolute disagreement
residual_sorted = residual.reindex(residual.abs().sort_values(ascending=False).index)

plt.figure(figsize=(18,6))

colors = ['tab:blue' if r > 0 else 'tab:red' for r in residual_sorted]

plt.bar(residual_sorted.index, residual_sorted.values, color=colors)
plt.xticks(rotation=90)

plt.ylabel("Human–Model Residual (z-scored false alarm rates)")
plt.title(
    f"Residuals Between Human and Model false alarm rates (Aligned) [r={r_hm:.2f}]\n"
    "Blue = Humans outperform model predictions | Red = Model overpredicts human performance"
)

plt.axhline(0, color='black', linewidth=1)
plt.tight_layout()
plt.savefig(savedir+"human_model_false_alarm_rate_residuals.png");
plt.show()



In [ ]:
aligned_humans

In [ ]:
aligned_model